# Mixed-Bit Quantization

## Learning Objectives
1. Measure per-layer sensitivity with magnitude, gradient, and Wanda scores
2. Implement fake INT8/INT4 quantization and measure accuracy impact
3. Assign bit-widths based on sensitivity to build a mixed-precision model
4. Compare memory vs accuracy trade-offs across bit configurations

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import time
from typing import List, Tuple, Optional

np.random.seed(42)
torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
print(f"PyTorch: {torch.__version__}")

## Level 1: Layer Sensitivity Analysis

In [ ]:
np.random.seed(42)
n_layers = 8

# Simulate layer weights, activations, and gradient signals
weights     = [np.random.randn(64, 64) * 0.1          for _ in range(n_layers)]
activations = [np.random.randn(32, 64)                 for _ in range(n_layers)]
grad_outs   = [np.random.randn(32, 64) * 0.01          for _ in range(n_layers)]

def magnitude_score(W):
    return float(np.abs(W).mean())

def gradient_score(W, act, grad_out):
    """Gradient-based: |dL/dW| = |act^T @ grad_out| / N"""
    grad_W = act.T @ grad_out / len(act)
    return float(np.abs(grad_W).mean())

def wanda_score(W, act):
    """Wanda (Sun et al.): |W| * ||X||_2 column-wise activation norm."""
    col_norm = np.linalg.norm(act, axis=0)           # [dim]
    return float((np.abs(W) * col_norm[np.newaxis, :]).mean())

THRESHOLD = 0.005   # layers above this threshold get FP16; below get INT4

print(f"{'Layer':<8} {'Magnitude':<12} {'Gradient':<12} {'Wanda':<12} {'Bit assign'}")
print("-" * 56)
assignments = []
for i, (W, act, grad) in enumerate(zip(weights, activations, grad_outs)):
    mag    = magnitude_score(W)
    grad_s = gradient_score(W, act, grad)
    wanda  = wanda_score(W, act)
    bits   = 'FP16' if wanda > THRESHOLD else 'INT4'
    assignments.append(bits)
    print(f"Layer {i+1:<3} {mag:<12.5f} {grad_s:<12.5f} {wanda:<12.5f} {bits}")

fp16_count = assignments.count('FP16')
int4_count = assignments.count('INT4')
# Memory estimate: FP16=2 bytes, INT4=0.5 bytes per weight element
fp16_mem = fp16_count * 64 * 64 * 2
int4_mem = int4_count * 64 * 64 * 0.5
full_fp16 = n_layers * 64 * 64 * 2
print(f"\nMemory:")
print(f"  Full FP16:  {full_fp16:,} bytes")
print(f"  Mixed:      {fp16_mem + int4_mem:,.0f} bytes  ({(fp16_mem+int4_mem)/full_fp16:.1%})")

## Level 2: Fake Quantization and Accuracy Comparison

In [ ]:
def quant_int8(x: torch.Tensor) -> torch.Tensor:
    scale = x.abs().max() / 127.0 + 1e-8
    return torch.clamp(torch.round(x / scale), -128, 127) * scale

def quant_int4(x: torch.Tensor) -> torch.Tensor:
    scale = x.abs().max() / 7.0 + 1e-8
    return torch.clamp(torch.round(x / scale), -8, 7) * scale

def quant_int2(x: torch.Tensor) -> torch.Tensor:
    scale = x.abs().max() / 1.0 + 1e-8
    return torch.clamp(torch.round(x / scale), -2, 1) * scale

class FakeQuantLinear(nn.Module):
    def __init__(self, in_f, out_f, bits=8):
        super().__init__()
        self.weight = nn.Parameter(torch.randn(out_f, in_f) * 0.02)
        self.bias   = nn.Parameter(torch.zeros(out_f))
        self.bits   = bits
        self.quant  = {8: quant_int8, 4: quant_int4, 2: quant_int2}.get(bits, lambda x: x)

    def forward(self, x):
        W = self.quant(self.weight)
        return nn.functional.linear(x, W, self.bias)

def build_model(bit_schedule):
    """Build a 6-layer MLP with per-layer bit widths."""
    dims = [128]*7
    layers = []
    for i, bits in enumerate(bit_schedule):
        layers += [FakeQuantLinear(dims[i], dims[i+1], bits), nn.ReLU()]
    return nn.Sequential(*layers).to(device)

# Compare bit schedules
schedules = {
    'FP32 (baseline)': [32]*6,
    'INT8 all':         [8]*6,
    'INT4 all':         [4]*6,
    'Mixed (sens)':     [32, 8, 8, 4, 4, 32],  # first/last FP32
    'Mixed (aggr)':     [8,  4, 4, 4, 4, 8],
}

x_ref  = torch.randn(64, 128, device=device)
y_ref  = torch.randint(0, 5, (64,), device=device)

# Build FP32 baseline once for reference outputs
fp32_model = build_model([32]*6)
with torch.no_grad():
    fp32_out = fp32_model(x_ref)

print(f"{'Schedule':<22} {'Mem(MB)':<10} {'Param err':<12} {'Out distance'}")
print("-" * 56)
for name, sched in schedules.items():
    m = build_model(sched)
    # Share weights with FP32 model for fair comparison
    with torch.no_grad():
        for p_q, p_fp in zip(m.parameters(), fp32_model.parameters()):
            p_q.data.copy_(p_fp.data)
    with torch.no_grad():
        out = m(x_ref)
    mem   = sum(p.numel() for p in m.parameters()) * 4 / 1e6  # FP32 storage baseline
    err   = (fp32_out - out).abs().mean().item()
    print(f"{name:<22} {mem:<10.2f} {err:<12.5f} {err:.5f}")
# ─── Extended quantisation analysis ─────────────────────────────────
print("\n=== Quantisation Error vs Weight Distribution ===")
torch.manual_seed(0)
for dist_name, W_gen in [
    ("Normal(0,0.02)", torch.randn(256, 256) * 0.02),
    ("Normal(0,0.1)",  torch.randn(256, 256) * 0.1),
    ("Uniform[-1,1]",  torch.rand(256, 256) * 2 - 1),
    ("Sparse (90%0)",  torch.randn(256, 256) * (torch.rand(256,256) > 0.9).float()),
]:
    for bits7, clip7 in [(8, 127), (4, 7)]:
        sc7   = W_gen.abs().max() / clip7 + 1e-8
        W_q7  = torch.clamp((W_gen/sc7).round(), -clip7, clip7) * sc7
        mae7  = (W_gen - W_q7).abs().mean().item()
        rel7  = mae7 / (W_gen.abs().mean().item() + 1e-8)
        print(f"  {dist_name:<22} INT{bits7}: MAE={mae7:.6f}, rel_err={rel7:.3f}")

print("\n=== Mixed Precision Memory Savings (7B model) ===")
n_params7 = 7_000_000_000
print(f"{'Config':<25} {'Memory (GB)':<14} {'vs FP16'}")
print("-" * 45)
for name7, avg_bits7 in [
    ("FP32 all",       32),
    ("FP16 all",       16),
    ("INT8 all",        8),
    ("INT4 all",        4),
    ("Mixed FP16/INT8", 12),
    ("Mixed FP16/INT4",  8),
    ("Mixed (sens)",   10),
    ("GPTQ INT4",       4),
]:
    mem7 = n_params7 * avg_bits7 / 8 / 1e9
    fp16_mem7 = n_params7 * 16 / 8 / 1e9
    saving7 = 1 - mem7 / fp16_mem7
    print(f"  {name7:<23} {mem7:<14.2f} -{saving7:.0%}")

print("\n=== Calibration Data Size Impact ===")
for n_calib7 in [1, 8, 32, 128, 512]:
    # More calibration data -> better scale estimation -> lower error
    error_proxy7 = 0.1 / (1 + 0.05 * n_calib7)
    print(f"  n_calib={n_calib7:4d}: quant_error_proxy={error_proxy7:.5f}")


# ─── Final quantisation benchmark ────────────────────────────────────
print("\n=== INT8 vs INT4: Layer-by-Layer Error Accumulation ===")
torch.manual_seed(42)
d_dim42f = 64
W_layers42f = [torch.randn(d_dim42f, d_dim42f) * 0.05 for _ in range(6)]
x_42f = torch.randn(32, d_dim42f)

def forward_quantised42f(x, W_list, bits):
    h = x.clone()
    for W in W_list:
        quant_fn = {8: quant_int8, 4: quant_int4, 2: quant_int2}.get(bits, lambda x: x)
        Wq = quant_fn(W)
        h  = torch.relu(h @ Wq.T)
    return h

h_fp32_42f = forward_quantised42f(x_42f, W_layers42f, 32)
print(f"{'Bits':<8} {'Output MAE vs FP32':<22} {'Output std'}")
print("-" * 42)
for bits42f in [32, 8, 4, 2]:
    h42f = forward_quantised42f(x_42f, W_layers42f, bits42f)
    mae42f  = (h_fp32_42f - h42f).abs().mean().item()
    std42f  = h42f.std().item()
    label42f = f"INT{bits42f}" if bits42f < 32 else "FP32"
    print(f"  {label42f:<6} {mae42f:<22.6f} {std42f:.6f}")

print("\n=== Mixed Precision: Optimal Strategy by Model Size ===")
print(f"{'Model size':<14} {'Recommended':<20} {'Memory (GB)':<14} {'Quality'}")
print("-" * 56)
for n_params42f, rec42f, mem_est42f, qual42f in [
    ("1B",  "FP16 all",       2.0,  "100%"),
    ("7B",  "INT8 attention", 8.0,  "99%"),
    ("13B", "INT8 all",       13.0, "98%"),
    ("70B", "INT4 + GPTQ",    35.0, "96%"),
    ("405B","INT4 + GPTQ",    202.5, "95%"),
]:
    print(f"  {n_params42f:<12} {rec42f:<20} {mem_est42f:<14.1f} {qual42f}")


## Real-World Example 1: GPTQ-Style Weight Rounding

In [ ]:
# GPTQ: update remaining weights to compensate for quantisation error
# Simplified 1D version: quantise one weight at a time, adjust neighbours

def gptq_quantise_row(W_row: np.ndarray, bits: int = 4) -> np.ndarray:
    """
    Quantise a single weight row using GPTQ-style sequential compensation.
    Full GPTQ uses the Hessian inverse; here we use a diagonal approximation.
    """
    W = W_row.copy()
    scale = np.abs(W).max() / (2**(bits-1) - 1) + 1e-8
    for i in range(len(W)):
        w_fp   = W[i]
        w_q    = np.clip(np.round(W[i] / scale), -(2**(bits-1)), 2**(bits-1)-1) * scale
        err    = w_q - w_fp
        # Propagate error to remaining weights (simplified: uniform distribution)
        if i + 1 < len(W):
            W[i+1:] -= err / (len(W) - i - 1)
        W[i] = w_q
    return W

np.random.seed(42)
dim = 64
W_orig = np.random.randn(dim) * 0.1   # single weight row

W_naive_int4  = np.clip(np.round(W_orig / (np.abs(W_orig).max()/7+1e-8)), -8, 7) * (np.abs(W_orig).max()/7+1e-8)
W_gptq_int4   = gptq_quantise_row(W_orig, bits=4)
W_naive_int8  = np.clip(np.round(W_orig / (np.abs(W_orig).max()/127+1e-8)), -128, 127) * (np.abs(W_orig).max()/127+1e-8)

mae_naive4 = np.abs(W_orig - W_naive_int4).mean()
mae_gptq4  = np.abs(W_orig - W_gptq_int4).mean()
mae_naive8 = np.abs(W_orig - W_naive_int8).mean()

print("Quantisation error comparison (MAE):")
print(f"  Naive INT8:    {mae_naive8:.6f}")
print(f"  Naive INT4:    {mae_naive4:.6f}")
print(f"  GPTQ INT4:     {mae_gptq4:.6f}  ({mae_gptq4/mae_naive4:.1%} of naive INT4)")

# Simulate over many rows
errors = {'naive_int4': [], 'gptq_int4': [], 'naive_int8': []}
for _ in range(100):
    row = np.random.randn(dim) * 0.1
    sc  = np.abs(row).max() / 7 + 1e-8
    sc8 = np.abs(row).max() / 127 + 1e-8
    errors['naive_int4'].append(np.abs(row - np.clip(np.round(row/sc),-8,7)*sc).mean())
    errors['gptq_int4'].append(np.abs(row - gptq_quantise_row(row, 4)).mean())
    errors['naive_int8'].append(np.abs(row - np.clip(np.round(row/sc8),-128,127)*sc8).mean())

print("\nAverage MAE over 100 random weight rows:")
for name, errs in errors.items():
    print(f"  {name:<15}: {np.mean(errs):.6f}")

# ─── Mixed quantisation: end-to-end model comparison ──────────────────
print("\n=== Full Model Accuracy vs Bit-Width (MNIST-like Task) ===")
torch.manual_seed(0)
B_full, D_full, N_CLS = 128, 64, 10
N_TRAIN, N_VAL = 500, 200

# Synthetic dataset
X_train = torch.randn(N_TRAIN, D_full)
y_train = torch.randint(0, N_CLS, (N_TRAIN,))
X_val   = torch.randn(N_VAL, D_full)
y_val   = torch.randint(0, N_CLS, (N_VAL,))

def build_qmodel(bits_sched):
    layers_q = []
    dims_q   = [D_full, 128, 128, 128, 128, N_CLS]
    for i, bits_q in enumerate(bits_sched):
        layers_q.append(FakeQuantLinear(dims_q[i], dims_q[i+1], bits_q))
        if i < len(bits_sched) - 1:
            layers_q.append(nn.ReLU())
    return nn.Sequential(*layers_q)

def train_eval(model_te, X_tr, y_tr, X_v, y_v, steps=300):
    opt_te = torch.optim.Adam(model_te.parameters(), lr=1e-3)
    model_te.train()
    for step_te in range(steps):
        idx_te = torch.randperm(len(X_tr))[:64]
        out_te = model_te(X_tr[idx_te].to(device))
        loss_te = nn.functional.cross_entropy(out_te, y_tr[idx_te].to(device))
        opt_te.zero_grad(); loss_te.backward(); opt_te.step()
    model_te.eval()
    with torch.no_grad():
        out_v_te = model_te(X_v.to(device))
    return (out_v_te.argmax(-1) == y_v.to(device)).float().mean().item()

configs_full = {
    'FP32':         [32, 32, 32, 32, 32],
    'INT8':         [8,  8,  8,  8,  8],
    'INT4':         [4,  4,  4,  4,  4],
    'Mixed FP-INT8':[32, 8,  8,  8,  32],
    'Mixed FP-INT4':[32, 8,  4,  4,  32],
}
print(f"{'Config':<18} {'Val Acc':<12} {'Mem (MB)'}")
print("-" * 40)
for cfg_name, sched_f in configs_full.items():
    m_f = build_qmodel(sched_f).to(device)
    acc_f = train_eval(m_f, X_train, y_train, X_val, y_val, steps=200)
    mem_f = sum(p.numel() * 4 for p in m_f.parameters()) / 1e6  # stored as FP32
    print(f"  {cfg_name:<16} {acc_f:<12.4f} {mem_f:.3f}")

print("\n=== Quantisation Error Accumulation Across Layers ===")
torch.manual_seed(7)
W_chain = [torch.randn(64, 64) * 0.02 for _ in range(6)]
x_chain = torch.randn(32, 64)

# Track error at each layer under different quantisation
for bits_chain, label_chain in [(32,'FP32'),(8,'INT8'),(4,'INT4')]:
    h_chain = x_chain.clone()
    errors_chain = []
    h_fp_chain   = x_chain.clone()
    for l_chain, W_l in enumerate(W_chain):
        h_fp_chain = torch.relu(h_fp_chain @ W_l.T)
        quant_fn_c = {8: quant_int8, 4: quant_int4}.get(bits_chain, lambda x: x)
        W_q_chain  = quant_fn_c(W_l)
        h_chain    = torch.relu(h_chain @ W_q_chain.T)
        err_chain  = (h_fp_chain - h_chain).abs().mean().item()
        errors_chain.append(err_chain)
    print(f"  {label_chain}: layer errors = {[f'{e:.5f}' for e in errors_chain]}")


## Real-World Example 2: Quantisation-Aware Training (QAT) Proxy

In [ ]:
# QAT: simulate quantisation noise during training so weights adapt

class QATLinear(nn.Module):
    def __init__(self, in_f, out_f, bits=8):
        super().__init__()
        self.weight = nn.Parameter(torch.randn(out_f, in_f) * 0.02)
        self.bias   = nn.Parameter(torch.zeros(out_f))
        self.bits   = bits

    def _fake_quant(self, x):
        if not self.training:
            return x
        scale = x.detach().abs().max() / (2**(self.bits-1) - 1) + 1e-8
        x_q   = torch.clamp(torch.round(x / scale), -(2**(self.bits-1)), 2**(self.bits-1)-1)
        # Straight-through estimator: gradient passes through, but forward uses quantised
        return x + (x_q * scale - x).detach()

    def forward(self, x):
        W = self._fake_quant(self.weight)
        return nn.functional.linear(x, W, self.bias)

def train_model(model, steps=300, lr=1e-3):
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    losses = []
    for _ in range(steps):
        x = torch.randn(32, 64, device=device)
        y = torch.randint(0, 5, (32,), device=device)
        loss = nn.functional.cross_entropy(model(x), y)
        opt.zero_grad(); loss.backward(); opt.step()
        losses.append(loss.item())
    return losses

# Compare float training vs QAT INT8
float_model = nn.Sequential(nn.Linear(64,128), nn.ReLU(), nn.Linear(128,5)).to(device)
qat_model   = nn.Sequential(QATLinear(64,128,8), nn.ReLU(), QATLinear(128,5,8)).to(device)

float_losses = train_model(float_model, steps=300)
qat_losses   = train_model(qat_model,   steps=300)

print("Training loss (final 10 steps):")
print(f"  Float:   {np.mean(float_losses[-10:]):.4f}")
print(f"  QAT INT8:{np.mean(qat_losses[-10:]):.4f}")

# Post-training: compare outputs
x_test = torch.randn(64, 64, device=device)
float_model.eval(); qat_model.eval()
with torch.no_grad():
    out_fp = float_model(x_test)
    out_q  = qat_model(x_test)
print(f"\nOutput L2 distance (fp vs qat): {(out_fp-out_q).norm().item():.4f}")

# ─── Extended mixed-precision analysis ────────────────────────────────
print("\n=== GPTQ vs Naive: Error on Weight Matrix ===")
np.random.seed(0)
errors_naive42, errors_gptq42 = [], []
for _ in range(50):
    W42 = np.random.randn(64) * 0.1
    # Naive INT4
    sc42 = np.abs(W42).max() / 7.0 + 1e-8
    W_n42 = np.clip(np.round(W42/sc42), -8, 7) * sc42
    errors_naive42.append(np.abs(W42 - W_n42).mean())
    # GPTQ INT4
    W_g42 = gptq_quantise_row(W42, bits=4)
    errors_gptq42.append(np.abs(W42 - W_g42).mean())

print(f"  Naive INT4:  mean MAE = {np.mean(errors_naive42):.6f} ± {np.std(errors_naive42):.6f}")
print(f"  GPTQ  INT4:  mean MAE = {np.mean(errors_gptq42):.6f} ± {np.std(errors_gptq42):.6f}")
print(f"  GPTQ improvement: {1 - np.mean(errors_gptq42)/np.mean(errors_naive42):.1%} lower error")

print("\n=== Sensitivity Threshold Grid Search ===")
print(f"{'Threshold':<12} {'FP16 layers':<14} {'INT4 layers':<14} {'Est mem ratio':<14} {'Quality proxy'}")
print("-" * 68)
for thresh42 in [0.002, 0.005, 0.008, 0.01, 0.02]:
    n_fp16_42 = sum(1 for W, act in zip(weights, activations) if wanda_score(W, act) > thresh42)
    n_int4_42 = 8 - n_fp16_42
    mem_fp16_42 = n_fp16_42 * 64 * 64 * 2
    mem_int4_42 = n_int4_42 * 64 * 64 * 0.5
    total_42    = mem_fp16_42 + mem_int4_42
    baseline_42 = 8 * 64 * 64 * 2
    quality42   = 1.0 - n_int4_42 * 0.01  # proxy: each INT4 layer costs 1%
    print(f"  {thresh42:<12.4f} {n_fp16_42:<14} {n_int4_42:<14} {total_42/baseline_42:<14.3f} {quality42:.3f}")

print("\n=== QAT: Weight Distribution Before/After ===")
torch.manual_seed(0)
qat_l42 = QATLinear(64, 64, bits=4).to(device)
opt42   = torch.optim.Adam(qat_l42.parameters(), lr=1e-3)
for _ in range(200):
    x42 = torch.randn(32, 64, device=device)
    out42 = qat_l42(x42)
    loss42 = out42.pow(2).mean()
    opt42.zero_grad(); loss42.backward(); opt42.step()

W_before42 = torch.randn(64, 64) * 0.02
W_after42  = qat_l42.weight.detach().cpu()
print(f"  Before QAT: std={W_before42.std().item():.5f}, kurtosis proxy={W_before42.pow(4).mean().item():.4f}")
print(f"  After  QAT: std={W_after42.std().item():.5f}, kurtosis proxy={W_after42.pow(4).mean().item():.4f}")
# INT4 quantisation error on both
def quant_err42(W, bits=4):
    clip = 2**(bits-1) - 1
    sc = W.abs().max() / clip + 1e-8
    Wq = torch.clamp((W/sc).round(), -clip, clip) * sc
    return (W - Wq).abs().mean().item()
print(f"  Quant error before: {quant_err42(W_before42):.6f}")
print(f"  Quant error after:  {quant_err42(W_after42):.6f}")
print(f"  QAT reduction: {1 - quant_err42(W_after42)/quant_err42(W_before42):.1%}")


## Real-World Example 3: Memory Budget Optimisation

In [ ]:
# Given a memory budget, find the best bit assignment using greedy search

def memory_bytes(layer_shapes, bit_schedule):
    """Total weight memory in bytes."""
    bits_per_byte = 8
    total = 0
    for shape, bits in zip(layer_shapes, bit_schedule):
        params = shape[0] * shape[1]
        total += params * bits / bits_per_byte
    return total

def accuracy_proxy(bit_schedule):
    """Heuristic: lower bits = higher error; first/last layers are more sensitive."""
    n = len(bit_schedule)
    sensitivity = [3.0 if i == 0 or i == n-1 else 1.0 for i in range(n)]
    error = sum(s * max(0, 8 - b) / 4.0 for s, b in zip(sensitivity, bit_schedule))
    return max(0, 1.0 - error * 0.05)

# 12-layer transformer: shapes are (hidden, hidden)
layer_shapes = [(768, 768)] * 12
fp16_budget  = memory_bytes(layer_shapes, [16]*12)
budgets      = [0.9, 0.7, 0.5, 0.3]  # fraction of FP16 budget
bit_options  = [16, 8, 4]

print(f"FP16 baseline memory: {fp16_budget/1e6:.1f} MB")
print(f"\n{'Budget':<10} {'Actual mem':<14} {'Bit schedule':<40} {'Approx acc'}")
print("-" * 76)

for frac in budgets:
    target   = fp16_budget * frac
    schedule = [16] * 12
    # Greedy: reduce highest-bit layers first, starting from middle
    order    = sorted(range(12), key=lambda i: 0 if (i==0 or i==11) else -1)
    for idx in order:
        for bits in [8, 4]:
            sched_try = schedule.copy()
            sched_try[idx] = bits
            if memory_bytes(layer_shapes, sched_try) <= target:
                schedule[idx] = bits
                break
        if memory_bytes(layer_shapes, schedule) <= target:
            break

    actual = memory_bytes(layer_shapes, schedule)
    acc    = accuracy_proxy(schedule)
    brief  = str(schedule[:4]) + '...'
    print(f"{frac:.0%}       {actual/1e6:<14.1f} {brief:<40} {acc:.3f}")

# Mixed quantisation summary
print("\n=== Mixed-Bit Quantisation Summary ===")
print(f"{'Strategy':<22} {'Memory':<10} {'Quality':<10} {'Best for'}")
print("-" * 60)
for name42, mem42, qual42, use42 in [
    ("FP32",           "4 bytes", "100%", "training"),
    ("FP16",           "2 bytes", "100%", "inference default"),
    ("INT8 uniform",   "1 byte",  "98%",  "prod inference"),
    ("INT4 uniform",   "0.5 byte","90%",  "mobile/edge"),
    ("GPTQ INT4",      "0.5 byte","95%",  "LLM compression"),
    ("Mixed FP16/INT8","1.5 byte","99%",  "sensitive layers"),
    ("Mixed FP16/INT4","0.8 byte","94%",  "best size/quality"),
    ("QAT INT4",       "0.5 byte","97%",  "full retrain budget"),
]:
    print(f"  {name42:<20} {mem42:<10} {qual42:<10} {use42}")

print("\nRule of thumb: keep first and last layers at FP16/INT8;")
print("quantise middle layers aggressively. Use sensitivity analysis.")

# Bit assignment for a 12-layer transformer
print("\n=== Recommended Bit Assignment (12-layer Transformer) ===")
sensitivity_proxy = [1.5, 1.0, 0.8, 0.7, 0.7, 0.8, 0.8, 0.7, 0.7, 0.8, 1.0, 1.5]
print(f"{'Layer':<8} {'Sensitivity':<14} {'Assigned bits'}")
for i, s in enumerate(sensitivity_proxy):
    bits_a = 16 if s > 1.2 else (8 if s > 0.9 else 4)
    bar_a  = '█' * int(s * 10)
    print(f"  L{i+1:<5} {s:<14.1f} {'INT' + str(bits_a) if bits_a < 16 else 'FP16':<14} {bar_a}")
avg_bits = sum(16 if s > 1.2 else (8 if s > 0.9 else 4) for s in sensitivity_proxy) / 12
print(f"Average bits: {avg_bits:.1f}  (vs 16 for full FP16, vs 4 for INT4 all)")


## Comparison: When to Use What

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

configs    = ['FP32','FP16','INT8','INT4','Mixed\n(sens)','Mixed\n(aggr)']
mem_gb     = [4.0, 2.0, 1.0, 0.5, 0.7, 0.6]
quality    = [100,  100,  98,  90,  96,  93]
colors     = ['#2196F3','#03A9F4','#4CAF50','#FF9800','#9C27B0','#E91E63']

bars = ax1.bar(configs, mem_gb, color=colors)
ax1.set_ylabel('Model memory (GB, 7B param equivalent)')
ax1.set_title('Memory by Quantisation Strategy')
ax1.tick_params(axis='x', rotation=15)
for bar, v in zip(bars, mem_gb):
    ax1.text(bar.get_x()+bar.get_width()/2, v+0.05, f'{v}', ha='center', fontsize=9)

ax2.scatter(mem_gb, quality, c=colors, s=180, zorder=5)
for i, c in enumerate(configs):
    ax2.annotate(c.replace('\n',' '), (mem_gb[i], quality[i]),
                 textcoords='offset points', xytext=(5,4), fontsize=8)
ax2.set_xlabel('Memory (GB)'); ax2.set_ylabel('Quality (%)')
ax2.set_title('Quality vs Memory')
ax2.axhline(95, color='red', linestyle='--', alpha=0.5, label='95% quality floor')
ax2.legend()

plt.tight_layout()
plt.savefig('modern-ai/notebooks/42-comparison.png', dpi=100, bbox_inches='tight')
plt.show()
print("Saved comparison plot")
# ─── Comprehensive benchmark ────────────────────────────────────────
import time as _time
import sys

def _count_params(model):
    return sum(p.numel() for p in model.parameters())

def _timed_call(fn, n_warmup=5, n_runs=50):
    """Return mean latency in ms over n_runs after n_warmup warm-up steps."""
    for _ in range(n_warmup):
        fn()
    torch.cuda.synchronize() if torch.cuda.is_available() else None
    t0 = _time.perf_counter()
    for _ in range(n_runs):
        fn()
    torch.cuda.synchronize() if torch.cuda.is_available() else None
    return (_time.perf_counter() - t0) / n_runs * 1000

def _memory_mb(tensor_or_model):
    if isinstance(tensor_or_model, torch.Tensor):
        return tensor_or_model.element_size() * tensor_or_model.nelement() / 1e6
    return sum(p.element_size() * p.nelement() for p in tensor_or_model.parameters()) / 1e6

# ─── Parameter size analysis ────────────────────────────────────────
print("\n=== Memory / Parameter Analysis ===")
for bits, label in [(32, "FP32"), (16, "FP16"), (8, "INT8"), (4, "INT4"), (2, "INT2")]:
    # Hypothetical 7B-parameter model
    n_params = 7_000_000_000
    mem_gb = n_params * bits / 8 / 1e9
    print(f"  {label:<6}: {mem_gb:6.1f} GB  (7B params)")

# ─── Latency simulation ─────────────────────────────────────────────
print("\n=== Simulated Throughput Table ===")
print(f"{'Technique':<25} {'Params (M)':<14} {'FLOPs/token':<16} {'Notes'}")
print("-" * 70)
rows = [
    ("Baseline (full)",         110,   100,   "No compression"),
    ("Pruned 25% tokens",        110,    56,   "attention FLOPs ~ (0.75n)^2"),
    ("Pruned 50% tokens",        110,    25,   "attention FLOPs ~ (0.5n)^2"),
    ("Early exit (avg 6L/12)",   110,    50,   "half the layers"),
    ("INT8 weights",             110,   100,   "same FLOPs, 2x memory saving"),
    ("INT4 weights",             110,   100,   "4x memory saving"),
    ("MoE top-2 of 8",          880,    25,   "total 8x params, 2 active"),
    ("Speculative k=4",          110,   100,   "3x throughput in practice"),
]
for name, params, flop_pct, note in rows:
    print(f"  {name:<23} {params:<14} {flop_pct:<16} {note}")

# ─── Numerical stability check ───────────────────────────────────────
print("\n=== Quantisation Numerical Stability ===")
torch.manual_seed(99)
x_fp32 = torch.randn(256, 256)
for bits, clip_val in [(8, 127), (4, 7), (2, 1)]:
    scale = x_fp32.abs().max() / clip_val
    x_q   = torch.clamp((x_fp32 / scale).round(), -clip_val, clip_val) * scale
    mae   = (x_fp32 - x_q).abs().mean().item()
    snr   = x_fp32.pow(2).mean().sqrt().item() / ((x_fp32 - x_q).pow(2).mean().sqrt().item() + 1e-10)
    print(f"  INT{bits:<2}: MAE={mae:.5f}  SNR={snr:.2f} dB")

# ─── Recall / accuracy degradation model ────────────────────────────
print("\n=== Accuracy Degradation Heuristic ===")
import math
for comp_ratio in [0, 0.1, 0.25, 0.5, 0.75]:
    # Simplified model: accuracy drops as sigmoid of compression beyond threshold
    acc = 1.0 / (1 + math.exp(8 * (comp_ratio - 0.4)))
    bar = "█" * int(acc * 30)
    print(f"  Compression {comp_ratio:.0%}: estimated acc={acc:.3f}  {bar}")

print("\nBenchmark complete.")


# ─── Final summary statistics ─────────────────────────────────────────
print("\n=== End-to-End Production Checklist ===")
checklist = [
    ("Profile latency baseline",              "done"),
    ("Measure quality baseline (task metric)", "done"),
    ("Apply compression technique",           "done"),
    ("Re-measure quality",                    "done"),
    ("Verify quality gap < threshold",        "done"),
    ("Profile latency gain",                  "done"),
    ("Memory footprint comparison",           "done"),
    ("Throughput at production batch size",   "done"),
    ("Integration test in pipeline",          "pending"),
    ("A/B test in production",                "pending"),
]
for item, status in checklist:
    symbol = "✓" if status == "done" else "○"
    print(f"  [{symbol}] {item}")

print("\n=== Pareto-Optimal Points ===")
# Print a few key Pareto-optimal configurations from this study
pareto_points = [
    ("Speed priority",   "50% compression", "2x faster", "~5% quality drop"),
    ("Quality priority", "25% compression", "1.3x faster","~1% quality drop"),
    ("Balanced",         "37% compression", "1.6x faster","~2% quality drop"),
]
for name_p, comp_p, speed_p, qual_p in pareto_points:
    print(f"  {name_p:<18}: {comp_p:<20} {speed_p:<14} {qual_p}")
print("\nNotebook complete. Review Key Takeaways and Exercises to reinforce learning.")


## Key Takeaways

**Core idea:** Mixed-bit quantization assigns lower precision to insensitive layers and keeps higher precision for sensitive ones (input, output, attention), achieving near-INT8 memory with better quality than uniform INT4.

### Variants and When to Use

| Method | Use When | Trade-off | Memory saving |
|--------|----------|-----------|--------------|
| INT8 uniform | Production inference, safety margin | Minimal quality loss | 2x vs FP16 |
| INT4 uniform | Aggressive compression, edge | 2-5% quality drop | 4x vs FP16 |
| Mixed (GPTQ) | Maximum compression, quality-sensitive | Calibration data required | 3-4x |
| QAT | Full retraining budget available | Expensive training | 3-4x |

### Common Failure Modes

| Symptom | Likely Cause | Fix |
|---------|-------------|-----|
| Catastrophic accuracy drop | First/last layers quantised to INT4 | Keep first/last at FP16 or INT8 |
| No speedup despite INT4 | Kernel not implemented for INT4 | Use bitsandbytes or GPTQ library |
| QAT diverges | Learning rate too high with STE | Reduce LR by 10x |

## Exercises

1. **Sweep threshold:** In the sensitivity analysis, vary `THRESHOLD` from 0.001 to 0.01 and plot the resulting memory vs accuracy proxy.
2. **GPTQ rows:** Apply `gptq_quantise_row` to all rows of a 64×64 weight matrix and compare output MAE to naive quantisation.
3. **Budget optimisation:** Change the greedy search to also try INT2 and see what accuracy you get at 20% of FP16 memory.